# Chapter 05
 Machine Learning for Business Analytics<br>
Concepts, Techniques, and Applications in Python<br>
by Galit Shmueli, Peter C. Bruce, Peter Gedeck, Nitin R. Patel

Publisher: Wiley; 2nd edition (2024) <br>
<!-- ISBN-13: 978-3031075650 -->

(c) 2024 Galit Shmueli, Peter C. Bruce, Peter Gedeck, Nitin R. Patel

The code needs to be executed in sequence.

Python packages and Python itself change over time. This can cause warnings or errors.
"Warnings" are for information only and can usually be ignored.
"Errors" will stop execution and need to be fixed in order to get results.

If you come across an issue with the code, please follow these steps

- Check the repository (https://gedeck.github.io/sdsa-code-solutions/) to see if the code has been upgraded. This might solve the problem.
- Report the problem using the issue tracker at https://github.com/gedeck/sdsa-code-solutions/issues
- Paste the error message into Google and see if someone else already found a solution

**Cell 1: Import libraries**  
This cell imports the necessary Python libraries: `matplotlib.pyplot` for plotting, `mlba` for data loading and utility functions, `pandas` for data manipulation, and various `sklearn` modules for linear regression, train‑test splitting, and classification metrics (`accuracy_score`, `auc`, `roc_curve`). The `%matplotlib inline` magic ensures that plots are displayed directly in the notebook.

In [ ]:
import matplotlib.pyplot as plt
import mlba
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, auc, roc_curve
from sklearn.model_selection import train_test_split
%matplotlib inline

**Cell 2: Load and prepare the Toyota Corolla data, fit a linear regression model, and print performance metrics**  
The Toyota Corolla dataset is loaded and reduced to the first 1000 rows for faster computation. Predictor variables are selected by excluding non‑numeric columns (`Price`, `Id`, `Model`, `Fuel_Type`, `Color`). The data is split into a training set (60%) and a holdout set (40%) using a fixed random state for reproducibility. A linear regression model is trained on the training data, and then performance is evaluated on both the training and holdout sets.  

**Interpretation of the printed statistics:**  
- **Training set:** Mean Error (ME) = 0.0000 (perfect mean, as expected for OLS), RMSE ≈ 1133, MAE ≈ 838, MPE ≈ –0.82%, MAPE ≈ 7.49%.  
- **Holdout set:** ME = 133.2 (slight positive bias), RMSE ≈ 1131, MAE ≈ 876, MPE ≈ 0.27%, MAPE ≈ 7.88%.  
The holdout MAE and MAPE are slightly higher than the training values, indicating a small amount of overfitting. However, the differences are modest, suggesting the model generalises reasonably well.

In [ ]:
# Load data frame and reduce it to 1000 rows for regression analysis
car_df = mlba.load_data('ToyotaCorolla.csv')
car_df = car_df.iloc[0:1000]

# create a list of predictor variables by removing output variables and text columns
excludeColumns = ('Price', 'Id', 'Model', 'Fuel_Type', 'Color')
predictors = [column for column in car_df.columns if column not in excludeColumns]
outcome = 'Price'

# partition data
X = car_df[predictors]
y = car_df[outcome]
train_X, holdout_X, train_y, holdout_y = train_test_split(X, y, test_size=0.4,
                                                          random_state=1)

# train linear regression model
reg = LinearRegression()
reg.fit(train_X, train_y)

# evaluate performance
# training
mlba.regressionSummary(y_true=train_y, y_pred=reg.predict(train_X))
# holdout
mlba.regressionSummary(y_true=holdout_y, y_pred=reg.predict(holdout_X))


The expression `[column for column in car_df.columns if column not in excludeColumns]` is a so-called list comprehension. It is a compact way to create a list from another list, with the condition that the element is not in the list of excluded columns. The way to read it is:
> for each column name in the list of column names,
>     if column is not in the list of excluded columns, include it in the new list.

**Cell 3: Visualise prediction errors – histograms and boxplots**  
Residuals (actual – predicted) are computed for the training and holdout sets. A figure with three subplots is created:  
- **Left:** histogram of training residuals (100 bins, range –6500 to 6500).  
- **Middle:** histogram of holdout residuals (same binning).  
- **Right:** boxplots comparing the two residual distributions.  

**Interpretation:** The histograms show that residuals for both sets are centred near zero and have similar spreads, though the holdout residuals exhibit a slightly longer tail on the positive side (visible in the boxplot). The boxplot confirms that the holdout residuals have a slightly higher median and wider interquartile range, again indicating a small degradation in performance on unseen data.

In [ ]:
pred_error_train = pd.DataFrame({
    'residual': train_y - reg.predict(train_X),
    'data set': 'training',
})
pred_error_holdout = pd.DataFrame({
    'residual': holdout_y - reg.predict(holdout_X),
    'data set': 'holdout',
})
boxdata_df = pd.concat([pred_error_train, pred_error_holdout])

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(8, 4))
common = {'bins': 100, 'range': [-6500, 6500]}
pred_error_train.hist(ax=axes[0], **common)
pred_error_holdout.hist(ax=axes[1], **common)
boxdata_df.boxplot(ax=axes[2], by='data set')

axes[0].set_title('training')
axes[1].set_title('holdout')
axes[2].set_title(' ')
axes[2].set_ylim(-6500, 6500)
plt.suptitle('')
plt.subplots_adjust(bottom=0.1, top=0.85, wspace=0.35)
plt.tight_layout()
plt.show()

**Cell 4: Gains chart and lift chart for the regression model**  
A DataFrame containing the holdout predictions and actual prices is created. The rows are sorted in descending order of predicted price. A gains chart (left) shows the cumulative actual price as a function of the number of cars (sorted by prediction). A lift chart (right) displays the lift – the ratio of the cumulative actual to the baseline (average) – for each decile.  

**Interpretation:** The gains curve rises steeply at the beginning, indicating that the model successfully ranks the most expensive cars at the top. The lift chart confirms that the top deciles have lift values > 1, meaning the model outperforms random selection. This is a common way to evaluate regression models for ranking (e.g., targeting high‑value customers).

In [ ]:
# sort the actual values in descending order of the prediction
df = pd.DataFrame({'predicted': reg.predict(holdout_X),
                   'actual': holdout_y})

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(7, 3.5))
ax = mlba.gainsChart(df, ranking='predicted', actual='actual', type='regression',
                     ax=axes[0])
ax.set_ylabel('Cumulative price')
ax.ticklabel_format(style='plain')

mlba.liftChart(df, ranking='predicted', actual='actual', ax=axes[1], labelBars=False)
plt.tight_layout()
plt.show()

**Cell 5: Quantifying the top‑10% performance**  
This cell calculates the average total price of a random 10% of the cars and compares it with the total price achieved when selecting the top 10% according to the model’s predictions.  

**Output interpretation:**  
- Number of cars in holdout set: 400.  
- 10% of cars = 40.  
- Average total price of 40 random cars ≈ 479,411.  
- Total price of the top‑40 cars ranked by prediction ≈ 815,569.  
- Ratio (lift) = 815,569 / 479,411 ≈ 1.70.  
This means the model’s top decile captures 1.7 times the value of a random sample, demonstrating substantial practical value for targeting high‑price cars.

In [ ]:
print(f'Number of cars: {len(df)}')
n = round(len(df) * 0.1)
print(f'Number of 10% of the cars: {n}')
average_10 = df['actual'].sum() * 0.1
print(f'Average sales price of 10% random cars: {average_10:.0f}')
# rank sales by predicted value (reverse order)
df = df.sort_values(by=['predicted'], ascending=False)
df['total'] = df['actual'].cumsum()
ranked_10 = df.iloc[n-1]['total']
print(f'Sale price of ranked 10%: {ranked_10:.0f}')
print(f'Ratio: {ranked_10 / average_10:.2f}')

**Cell 6: Scatter plots illustrating separation in classification**  
Two datasets are loaded: `RidingMowers` and `UniversalBank`. The top panel shows Income vs. Lot Size for riding‑mower owners/non‑owners; the classes are well separated (high separation). The bottom panel shows Income vs. average credit card spending for bank customers who did/did not accept a personal loan; here the classes overlap heavily (low separation), and a log‑log scale is used to spread the points.  

**Interpretation:** The contrast highlights how the degree of class separation affects the potential performance of a classifier. When classes are well separated, even simple models can achieve high accuracy; when overlap is large, more sophisticated models and careful threshold selection are required.

In [ ]:
mower_df = mlba.load_data('RidingMowers.csv')
universal_df = mlba.load_data('UniversalBank.csv')

fig, axes = plt.subplots(figsize=(6, 8), nrows=2)
ax=axes[0]
subset = mower_df.loc[mower_df['Ownership']=='Owner']
ax.scatter(subset.Income, subset.Lot_Size, marker='o',
    label='Owner', color='C1')
subset = mower_df.loc[mower_df['Ownership']=='Nonowner']
ax.scatter(subset.Income, subset.Lot_Size, marker='o',
    label='Nonowner', color='C0')
ax.set_xlabel('Income ($000s)')
ax.set_ylabel('Lot size (000s sqft)')
ax.set_title('High level of separation')
ax.legend()

ax = axes[1]
subset = universal_df.loc[universal_df['Personal Loan'] == 0]
subset.plot.scatter(x='Income', y='CCAvg', ax=ax, alpha=0.8,
                    color='C1', label='Nonacceptor')
subset = universal_df.loc[universal_df['Personal Loan'] == 1]
subset.plot.scatter(x='Income', y='CCAvg', ax=ax, alpha=0.8,
                    color='C0', label='Acceptor')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Annual income ($000s)')
ax.set_ylabel('Monthly credit card average spending ($000s)')
ax.set_title('Low level of separation')
ax.legend()
plt.tight_layout()
plt.show()

**Cell 7: Effect of cutoff on classification – cutoff = 0.5**  
The `ownerExample` dataset contains predicted probabilities of being an owner. Using a cutoff of 0.5, predictions are converted to class labels, and a confusion matrix with accuracy is printed.  

**Output:** Accuracy = 0.8750, with 10 non‑owners correctly classified, 2 misclassified as owners, 1 owner misclassified as non‑owner, and 11 owners correctly classified.

In [ ]:
owner_df = mlba.load_data('ownerExample.csv')

## cutoff = 0.5
predicted = ['owner' if p > 0.5 else 'nonowner' for p in owner_df.Probability]
mlba.classificationSummary(y_true=owner_df.Class, y_pred=predicted)

**Cell 8: Cutoff = 0.25**  
Lowering the cutoff makes the classifier more liberal in predicting the positive class (`owner`). The confusion matrix shows 8 true non‑owners, 4 false owners, 1 false non‑owner, and 11 true owners. Accuracy drops to 0.7917.

In [ ]:
## cutoff = 0.25
predicted = ['owner' if p > 0.25 else 'nonowner' for p in owner_df.Probability]
mlba.classificationSummary(y_true=owner_df.Class, y_pred=predicted)

**Cell 9: Cutoff = 0.75**  
Raising the cutoff makes the classifier more conservative. The confusion matrix shows 11 true non‑owners, 1 false owner, 5 false non‑owners, and 7 true owners. Accuracy further decreases to 0.75.

In [ ]:
## cutoff = 0.75
predicted = ['owner' if p > 0.75 else 'nonowner' for p in owner_df.Probability]
mlba.classificationSummary(y_true=owner_df.Class, y_pred=predicted)

**Cell 10: Accuracy and overall error as functions of cutoff**  
Using the `liftExample` dataset (binary classification with predicted probabilities), accuracy is computed for cutoffs from 0.0 to 1.0 in steps of 0.1. Both accuracy and overall error (1 – accuracy) are plotted against the cutoff.  

**Interpretation:** Accuracy peaks around a cutoff of 0.5, which is the default. The overall error curve is the mirror image. This demonstrates that the choice of cutoff affects performance; the optimal cutoff may not always be 0.5, especially when classes are imbalanced or misclassification costs differ.

In [ ]:
df = mlba.load_data('liftExample.csv')

cutoffs = [i * 0.1 for i in range(11)]
accT = []
for cutoff in cutoffs:
    predicted = [1 if p > cutoff else 0 for p in df.prob]
    accT.append(accuracy_score(df.actual, predicted))

fig, ax = plt.subplots()
fig.set_size_inches(6, 4)
ax.plot(cutoffs, accT, '-', label='Accuracy')
ax.plot(cutoffs, [1 - acc for acc in accT], '--', label='Overall error')
ax.set_ylim(0, 1)
ax.set_xlabel('Cutoff value')
ax.legend()
plt.tight_layout()
plt.show()

**Cell 11: ROC curve and AUC**  
For the same `liftExample` data, the false positive rate (1 – specificity) and true positive rate (sensitivity) are computed at various thresholds, and the ROC curve is plotted. The area under the curve (AUC) is calculated.  

**Interpretation:** The ROC curve lies well above the diagonal, indicating good discriminative power. The AUC of 0.8889 confirms that the model has strong separation ability; an AUC of 0.5 would correspond to random guessing.

In [ ]:
# compute ROC curve and AUC
fpr, tpr, _ = roc_curve(df.actual, df.prob)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=[5, 5])
ax.plot(fpr, tpr, color='darkorange',
        lw=2, label=f'ROC curve (area = {roc_auc:0.4f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0, 1.05])
ax.set_xlabel('False positive rate (1 - specificity)')
ax.set_ylabel('True positive rate (sensitivity)')
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Cell 12: Precision, recall, and F1‑score across cutoffs**  
Using the same data, precision, recall, and F1‑score are computed for each unique probability value as cutoff. The three metrics are plotted against the cutoff.  

**Interpretation:** Precision and recall trade off: as the cutoff increases, precision tends to rise (fewer false positives) while recall falls (more false negatives). The F1‑score, the harmonic mean of precision and recall, peaks around the cutoff where the two curves cross, indicating a balanced choice.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
import numpy as np
df = mlba.load_data('liftExample.csv')

metrics = []
cutoffs = np.arange(0, 1, 0.05)
cutoffs = sorted(df['prob'].unique())
for cutoff in cutoffs:
    df['predicted'] = [1 if p >= cutoff else 0 for p in df['prob']]
    precision, recall, fscore, _ = precision_recall_fscore_support(
        y_true=df['actual'], y_pred=df['predicted'], zero_division=np.nan)
    metrics.append({
        'Cutoff': cutoff,
        'Precision': precision[1],
        'Recall': recall[1],
        'F1-score': fscore[1],
    })
metrics = pd.DataFrame(metrics)
fig, ax = plt.subplots(figsize=[6, 4])
metrics.plot(x='Cutoff', y='F1-score', ax=ax, linewidth=3)
metrics.plot(x='Cutoff', y='Precision', ax=ax)
metrics.plot(x='Cutoff', y='Recall', ax=ax)
plt.tight_layout()
plt.show()

**Cell 13: Gains charts for classification (with and without counts)**  
Two gains charts are generated for the `liftExample` data. The left chart shows the cumulative percentage of positive cases as a function of the percentage of the population sorted by predicted probability. The right chart adds the actual counts on the secondary axis.  

**Interpretation:** The gains curve rises steeply, indicating that the model concentrates positives in the top deciles. The chart with counts shows, for example, that the top 20% of the population (by predicted probability) contains about 80% of the actual positives – a strong performance.

In [ ]:
df = mlba.load_data('liftExample.csv')
fig, axes = plt.subplots(ncols=2, figsize=(8, 4))
mlba.gainsChart(df, ranking='prob', actual='actual', ax=axes[0])
mlba.gainsChart(df, ranking='prob', actual='actual', show_counts=True, ax=axes[1])
plt.tight_layout()
plt.show()

**Cell 14: Lift chart for classification**  
A lift chart is produced for the same data. Lift is defined as the ratio of the cumulative percentage of positives in each decile to the overall percentage of positives.  

**Interpretation:** The first decile has a lift well above 2, meaning that in the top 10% of the population (ranked by predicted probability), the concentration of actual positives is more than twice the overall rate. As we move to lower deciles, lift declines, eventually dropping below 1.

In [ ]:
mlba.liftChart(df, ranking='prob', actual='actual', labelBars=False)
plt.tight_layout()
plt.show()